# UrduStack — Train Risk Scorer (LoRA XLM-RoBERTa) [HARDENED]

Run this notebook in Google Colab free-tier GPU (T4).

**What it does**
1. **Builds a frequency map** from Roman-Urdu-Parl (6.37M parallel sentences)
2. **Downloads 4 datasets** from Hugging Face automatically
3. **Merges** them into a combined training set with class balance audit
4. **Fine-tunes** `xlm-roberta-base` using LoRA (5 epochs)
5. **Auto-downloads** the trained model as a zip file to your computer
6. **Evaluates** Whisper on Urdu speech benchmark (WER baseline)
7. **Launches** a live Gradio demo with a public URL

**IMPORTANT BEFORE YOU START:**
- Make sure GPU is enabled: Runtime > Change runtime type > T4 GPU
- Allow browser popups for Colab (needed for auto-download)
- Do NOT close or switch away from the tab during training

**Expected total runtime on free Colab T4: ~80-120 minutes**

In [ ]:
import sys, torch

if not torch.cuda.is_available():
    print("STOP: No GPU detected!")
    print("Go to: Runtime > Change runtime type > T4 GPU")
    print("Then re-run all cells from the top.")
    sys.exit(1)

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print("GPU check passed.")

In [ ]:
import subprocess, time

def pip_install(packages, attempt=1):
    print(f"Attempt {attempt}: installing packages...")
    subprocess.run(["pip", "uninstall", "-y", "torchao"],
                    capture_output=True)
    result = subprocess.run(
        ["pip", "install", "-q", "-U"] + packages,
        capture_output=True, text=True
    )
    return result.returncode == 0

packages = [
    "transformers>=4.46.0",
    "datasets>=3.1.0",
    "peft>=0.13.2",
    "accelerate>=1.1.0",
    "openai-whisper>=20231117",
    "scikit-learn",
    "pandas",
]

ok = pip_install(packages, 1)
if not ok:
    print("First attempt failed, retrying in 5s...")
    time.sleep(5)
    ok = pip_install(packages, 2)
if not ok:
    raise RuntimeError("pip install failed after 2 attempts")

import peft, transformers, datasets
print(f"peft={peft.__version__}  transformers={transformers.__version__}  datasets={datasets.__version__}")

for mod_name in ['torch', 'sklearn', 'pandas', 'whisper']:
    try:
        __import__(mod_name)
    except ImportError:
        raise RuntimeError(f"CRITICAL: {mod_name} is not installed")

print("All dependencies verified.")

In [ ]:
import os, shutil, subprocess, time

if os.path.exists('UrduStack/UrduStack'):
    shutil.rmtree('UrduStack/UrduStack')
    print('Removed nested UrduStack/UrduStack')

if os.path.exists('UrduStack'):
    os.chdir('UrduStack')
    subprocess.run(['git', 'pull'])
else:
    for attempt in range(1, 4):
        result = subprocess.run(
            ['git', 'clone', 'https://github.com/munazat/UrduStack.git'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            break
        print(f"Clone failed (attempt {attempt}/3): {result.stderr.strip()}")
        if attempt < 3:
            time.sleep(5)
    else:
        raise RuntimeError("Could not clone repo after 3 attempts")
    os.chdir('UrduStack')

print(f"Working directory: {os.getcwd()}")

for f in ['scripts/train_risk_model.py', 'scripts/build_normalizer_map.py',
           'scripts/generate_scam_data.py', 'playground.py']:
    if not os.path.exists(f):
        raise RuntimeError(f"CRITICAL: missing file: {f}")

print("Repo ready. All required scripts found.")

In [ ]:
import os, gc, json, time
from datasets import load_dataset

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

parl_csv = 'data/raw/roman_urdu_parl.csv'

if not os.path.exists(parl_csv):
    print('Downloading Roman-Urdu-Parl from Hugging Face (6.37M rows)...')
    for attempt in range(1, 4):
        try:
            ds = load_dataset('Mavkif/Roman-Urdu-Parl-split', split='train')
            df_parl = ds.to_pandas()
            df_parl.to_csv(parl_csv, index=False)
            print(f'Downloaded {len(df_parl)} rows')
            del df_parl
            gc.collect()
            break
        except Exception as e:
            print(f"Download failed (attempt {attempt}/3): {e}")
            if attempt == 3:
                raise RuntimeError("Could not download Roman-Urdu-Parl after 3 attempts")
            time.sleep(10)
else:
    print(f'{parl_csv} already present, skipping download.')

print('Building frequency map (~5-10 minutes)...')
!python scripts/build_normalizer_map.py \
  --input data/raw/roman_urdu_parl.csv \
  --output data/processed/roman_urdu_freq.json \
  --min_count 2 \
  --chunk_size 500000

if os.path.exists('data/processed/roman_urdu_freq.json'):
    with open('data/processed/roman_urdu_freq.json', 'r', encoding='utf-8') as f:
        freq_map = json.load(f)
    print(f'Frequency map: {len(freq_map)} mappings')
    sample = list(freq_map.items())[:5]
    for r, u in sample:
        print(f'  {r} -> {u}')
    if len(freq_map) < 10000:
        print(f"WARNING: frequency map is small ({len(freq_map)} entries)")
        print("Training will still work, normalization may be less effective.")
else:
    print('WARNING: frequency map not created. Non-critical, continuing.')

del freq_map
gc.collect()
print('Done.')

In [ ]:
import os, gc, time
from datasets import load_dataset

os.makedirs('data/raw', exist_ok=True)

if not os.path.exists('data/raw/PURUTT.csv'):
    print('Downloading Roman-Urdu-Toxic-Corpus (72.7k rows)...')
    for attempt in range(1, 4):
        try:
            ds = load_dataset(
                'hafiz-hassaan-saeed/Roman-Urdu-Toxic-Corpus',
                split='train'
            )
            df = ds.to_pandas()
            if 'Roman_Urdu' in df.columns and 'Toxic' in df.columns:
                df = df.rename(columns={'Roman_Urdu': 'text', 'Toxic': 'label'})
            elif 'text' not in df.columns or 'label' not in df.columns:
                raise RuntimeError(
                    f"Unexpected columns: {list(df.columns)}. "
                    f"Expected 'Roman_Urdu'/'Toxic' or 'text'/'label'."
                )
            df.to_csv('data/raw/PURUTT.csv', index=False)
            print(f'Downloaded {len(df)} rows')
            del df
            gc.collect()
            break
        except Exception as e:
            print(f"Download failed (attempt {attempt}/3): {e}")
            if attempt == 3:
                raise RuntimeError("Could not download PURUTT after 3 attempts")
            time.sleep(10)
else:
    print('PURUTT.csv already present, skipping.')

import pandas as pd
purutt = pd.read_csv('data/raw/PURUTT.csv')
assert 'text' in purutt.columns and 'label' in purutt.columns, \
    f"PURUTT missing columns. Found: {list(purutt.columns)}"
assert len(purutt) > 1000, f"PURUTT too small: {len(purutt)} rows"
print(f"Verified: {len(purutt)} rows with correct columns.")

In [ ]:
import os, gc
import pandas as pd

frames = []

# --- PURUTT (required) ---
purutt = pd.read_csv('data/raw/PURUTT.csv')[['text', 'label']].dropna()
purutt['label'] = purutt['label'].astype(int)
print(f"PURUTT: {len(purutt)} rows | toxic={purutt['label'].sum()} clean={(purutt['label']==0).sum()}")
frames.append(purutt)

# --- Hate Speech (optional) ---
try:
    from datasets import load_dataset
    hate_ds = load_dataset(
        'community-datasets/roman_urdu_hate_speech',
        'Coarse_Grained', split='train'
    )
    hate_df = hate_ds.to_pandas().rename(columns={'tweet': 'text'})
    hate_df['label'] = 1 - hate_df['label'].astype(int)
    hate_df = hate_df[['text', 'label']].dropna()
    print(f"Hate Speech: {len(hate_df)} rows | toxic={hate_df['label'].sum()} clean={(hate_df['label']==0).sum()}")
    frames.append(hate_df)
except Exception as e:
    print(f"WARNING: hate speech dataset unavailable: {e}")

# --- Spam (optional) ---
try:
    from datasets import load_dataset
    spam_ds = load_dataset('hamza-amin/urdu-spam-dataset', split='train')
    spam_df = spam_ds.to_pandas()[['text', 'label']].dropna()
    spam_df['label'] = spam_df['label'].astype(int)
    print(f"Spam: {len(spam_df)} rows | spam={spam_df['label'].sum()} clean={(spam_df['label']==0).sum()}")
    frames.append(spam_df)
except Exception as e:
    print(f"WARNING: spam dataset unavailable: {e}")

# --- Synthetic Scam ---
print("Generating synthetic scam data...")
!python scripts/generate_scam_data.py --output data/raw/synthetic_scam.csv --n_samples 1500 --seed 42

if os.path.exists('data/raw/synthetic_scam.csv'):
    scam_gen = pd.read_csv('data/raw/synthetic_scam.csv')[['text', 'label']].dropna()
    scam_gen['label'] = scam_gen['label'].astype(int)
    print(f"Synthetic Scam: {len(scam_gen)} rows")
    frames.append(scam_gen)
else:
    print("WARNING: synthetic scam data not generated")

if len(frames) < 1:
    raise RuntimeError("No datasets loaded. Cannot train.")

combined = pd.concat(frames, ignore_index=True)
combined = combined.drop_duplicates(subset=['text']).sample(frac=1, random_state=42).reset_index(drop=True)
combined.to_csv('data/raw/combined_risk.csv', index=False)

total = len(combined)
toxic = combined['label'].sum()
clean = (combined['label'] == 0).sum()
print(f"{'='*50}")
print(f"Combined dataset: {total} rows")
print(f"  Toxic/Spam (label=1): {toxic} ({toxic/total*100:.1f}%)")
print(f"  Clean      (label=0): {clean} ({clean/total*100:.1f}%)")
if toxic / max(clean, 1) < 0.3 or toxic / max(clean, 1) > 3.0:
    print("  WARNING: class imbalance detected, class-weighted loss will compensate.")
print(f"{'='*50}")

if total < 10000:
    raise RuntimeError(f"Combined dataset too small ({total} rows). PURUTT download may have failed.")

gc.collect()

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

combined = pd.read_csv('data/raw/combined_risk.csv')

print(f"Total samples: {len(combined):,}")
label_counts = combined['label'].value_counts().sort_index()
for label, count in label_counts.items():
    pct = count / len(combined) * 100
    tag = "toxic/spam" if label == 1 else "clean"
    print(f"  label={label} ({tag}): {count:,} ({pct:.1f}%)")

combined['text_len'] = combined['text'].str.len()
print(f"\nText length stats:")
print(f"  Mean:   {combined['text_len'].mean():.0f} chars")
print(f"  Median: {combined['text_len'].median():.0f} chars")
print(f"  Min:    {combined['text_len'].min()} chars")
print(f"  Max:    {combined['text_len'].max()} chars")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels_for_pie = []
for label_val in sorted(label_counts.index):
    tag = "Toxic/Spam" if label_val == 1 else "Clean"
    labels_for_pie.append(f"{tag} (label={label_val})\n{label_counts[label_val]:,}")

axes[0].pie(
    label_counts.values,
    labels=labels_for_pie,
    autopct='%1.1f%%',
    colors=['#4CAF50', '#F44336'],
    startangle=90,
)
axes[0].set_title('Label Distribution')

for label_val, color, tag in [(0, '#4CAF50', 'Clean'), (1, '#F44336', 'Toxic/Spam')]:
    subset = combined[combined['label'] == label_val]['text_len']
    axes[1].hist(subset, bins=50, alpha=0.6, label=f'{tag} (n={len(subset):,})', color=color)
axes[1].set_xlabel('Text Length (chars)')
axes[1].set_ylabel('Count')
axes[1].set_title('Text Length by Class')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/processed/class_balance_audit.png', dpi=100, bbox_inches='tight')
plt.show()

toxic_ratio = label_counts[1] / len(combined)
if toxic_ratio < 0.1:
    print("\nWARNING: severe imbalance, toxic < 10%. Class-weighted loss will compensate.")
elif toxic_ratio > 0.9:
    print("\nWARNING: severe imbalance, clean < 10%.")
else:
    print(f"\nClass balance OK: toxic ratio = {toxic_ratio:.1%}")

In [ ]:
import os, subprocess, sys

print("=" * 60)
print("ACTIVE LEARNING: Consume Feedback")
print("=" * 60)
print()
print("This cell closes the active learning loop:")
print("  1. Reads data/feedback.csv (corrections from the demo)")
print("  2. Filters for high-confidence corrections")
print("  3. Merges them into the training set")
print("  4. The next training run uses the improved data")
print()

feedback_path = 'data/feedback.csv'
combined_path = 'data/raw/combined_risk.csv'

# Show pre-training dataset size
import pandas as pd
if os.path.exists(combined_path):
    before = len(pd.read_csv(combined_path))
    print(f"Training set BEFORE feedback: {before} rows")
else:
    before = 0
    print("WARNING: combined_risk.csv not found. Run dataset cells first.")

if os.path.exists(feedback_path):
    fb_df = pd.read_csv(feedback_path)
    print(f"Feedback entries: {len(fb_df)}")
    corrections = fb_df[fb_df['correct_label'] != 'correct']
    print(f"  Corrections (not 'correct'): {len(corrections)}")

    # Run consume_feedback.py with merge
    result = subprocess.run(
        [sys.executable, 'scripts/consume_feedback.py',
         '--merge_into', combined_path],
        capture_output=True, text=True, cwd='/content/UrduStack'
    )
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr[-300:])

    if os.path.exists(combined_path):
        after = len(pd.read_csv(combined_path))
        print(f"\nTraining set AFTER feedback: {after} rows")
        if before > 0:
            added = after - before
            print(f"  Net new rows added: {added}")
        print("\nFeedback merged. Next training run will include corrections.")
    else:
        print("Merge target not created.")
else:
    print(f"No feedback file at {feedback_path}.")
    print("To generate feedback:")
    print("  1. Run the Gradio demo (last cell)")
    print("  2. Analyze some text")
    print("  3. Use the feedback dropdown to correct wrong predictions")
    print("  4. Re-run this cell")
    print()
    print("Or create sample feedback for demo purposes:")
    print("  Run the cell below to generate sample feedback.csv")

In [ ]:
import subprocess, time

print("=" * 60)
print("TRAINING STARTING")
print("LoRA XLM-RoBERTa | 70k train + 5k val + 5k test | 5 epochs")
print("Expected runtime: 60-90 minutes on T4 GPU")
print("=" * 60)
print()
print("IMPORTANT: Keep this tab active. Do not close or switch tabs.")
print("Auto-download will trigger when training finishes.")
print()

start_time = time.time()

result = subprocess.run([
    'python', 'scripts/train_risk_model.py',
    '--data_path', 'data/raw/combined_risk.csv',
    '--output_dir', 'models/risk_lora',
    '--max_samples', '70000',
    '--val_samples', '5000',
    '--test_samples', '5000',
    '--num_epochs', '5',
    '--batch_size', '16',
])

elapsed = time.time() - start_time
print(f"\nTraining finished in {elapsed/60:.1f} minutes (exit code: {result.returncode})")

if result.returncode != 0:
    print()
    print("=" * 60)
    print("TRAINING FAILED")
    print("=" * 60)
    print("Try these fixes:")
    print("1. Run this cell again (transient GPU error)")
    print("2. Reduce --max_samples to 50000")
    print("3. Reduce --num_epochs to 3")
    raise SystemExit(1)

print("\nTraining completed successfully!")

In [ ]:
import os, zipfile, time
from google.colab import files

print("=" * 60)
print("VERIFYING TRAINING OUTPUT")
print("=" * 60)

# --- Check temperature.txt ---
temp_path = 'models/temperature.txt'
if not os.path.exists(temp_path):
    raise RuntimeError(f"CRITICAL: {temp_path} does not exist. Training did not complete.")
temp_content = open(temp_path).read().strip()
if not temp_content:
    raise RuntimeError(f"CRITICAL: {temp_path} is empty. Training did not complete.")
try:
    temp_val = float(temp_content)
    print(f"  temperature.txt: {temp_val:.4f} OK")
except ValueError:
    raise RuntimeError(f"CRITICAL: {temp_path} has invalid content: '{temp_content}'")

# --- Check adapter files ---
adapter_dir = 'models/risk_lora'
if not os.path.isdir(adapter_dir):
    raise RuntimeError(f"CRITICAL: {adapter_dir}/ does not exist.")

required_files = ['adapter_config.json', 'adapter_model.safetensors']
alt_files = ['adapter_config.json', 'adapter_model.bin']

has_safetensors = all(
    os.path.exists(os.path.join(adapter_dir, f)) for f in required_files
)
has_bin = all(
    os.path.exists(os.path.join(adapter_dir, f)) for f in alt_files
)

if not has_safetensors and not has_bin:
    actual = os.listdir(adapter_dir)
    raise RuntimeError(
        f"CRITICAL: adapter files missing. Found: {actual}"
    )
print(f"  Adapter files: OK")

# --- Check tokenizer ---
tokenizer_files = [
    os.path.join(adapter_dir, f)
    for f in ['tokenizer.json', 'tokenizer_config.json', 'vocab.txt',
              'sentencepiece.bpe.model']
    if os.path.exists(os.path.join(adapter_dir, f))
]
if tokenizer_files:
    print(f"  Tokenizer: {len(tokenizer_files)} files OK")
else:
    print("  WARNING: no tokenizer files found in adapter dir")

# --- Create zip ---
zip_path = '/content/urdustack_model.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(temp_path, 'temperature.txt')
    for root, dirs, filenames in os.walk(adapter_dir):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            arcname = os.path.join(
                'risk_lora',
                os.path.relpath(filepath, adapter_dir)
            )
            zf.write(filepath, arcname)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\nZip created: {zip_path} ({zip_size_mb:.1f} MB)")

# --- Auto-download ---
print("\n" + "=" * 60)
print("DOWNLOADING MODEL...")
print("=" * 60)
print()
print("If download doesn't start automatically:")
print("  1. Check for a browser popup blocker and allow it")
print("  2. Run the cell below (Manual Download) as backup")
print("  3. Or use the Files panel (left sidebar) to download")
print()

try:
    files.download(zip_path)
    print("Download triggered!")
except Exception as e:
    print(f"Auto-download failed: {e}")
    print("Use the Manual Download cell below, or:")
    print(f"  Files panel > navigate to {zip_path} > right-click > Download")

print()
print("=" * 60)
print("SAVE YOUR MODEL NOW!")
print("Colab can disconnect at any time.")
print("Make sure the zip file is on your computer before continuing.")
print("=" * 60)

In [ ]:
from google.colab import files
import os

download_targets = [
    '/content/urdustack_model.zip',
    'models/temperature.txt',
    'models/risk_lora/adapter_config.json',
    'models/risk_lora/adapter_model.safetensors',
    'models/risk_lora/adapter_model.bin',
    'models/risk_lora/tokenizer.json',
    'models/risk_lora/tokenizer_config.json',
    'models/risk_lora/sentencepiece.bpe.model',
]

print("Downloading files (allow popups if prompted)...")
print()

for f in download_targets:
    if os.path.exists(f):
        try:
            files.download(f)
            size_kb = os.path.getsize(f) / 1024
            print(f"  Downloaded: {f} ({size_kb:.0f} KB)")
            import time
            time.sleep(1)
        except Exception as e:
            print(f"  FAILED: {f} -- {e}")
    else:
        print(f"  Skipped (not found): {f}")

print()
print("If downloads are blocked:")
print("  1. Click the folder icon in the left sidebar")
print("  2. Navigate to the file path shown above")
print("  3. Right-click > Download")

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'jiwer', 'soundfile'], capture_output=True)

import torch, gc
import numpy as np

print("Loading Whisper base model...")
device = "cuda" if torch.cuda.is_available() else "cpu"

import whisper
whisper_model = whisper.load_model("base", device=device)
print(f"Whisper loaded on {device}")

from datasets import load_dataset
from jiwer import wer, cer

hypotheses = []
references = []
n_eval = 50

def evaluate_samples(eval_iter, n, source_name):
    hyps, refs = [], []
    for i, sample in enumerate(eval_iter):
        if i >= n:
            break
        audio = sample["audio"]
        if isinstance(audio, dict):
            audio_array = np.array(audio["array"], dtype=np.float32)
        else:
            audio_array = np.array(audio, dtype=np.float32)
        if len(audio_array.shape) > 1:
            audio_array = audio_array.mean(axis=1)
        result = whisper_model.transcribe(
            audio_array, language="ur", fp16=(device == "cuda")
        )
        hyp = result.get("text", "").strip()
        ref = sample.get("text", "").strip()
        if ref:
            hyps.append(hyp)
            refs.append(ref)
        if (i + 1) % 10 == 0:
            print(f"  [{source_name}] Processed {i+1}/{n}...")
    return hyps, refs

# Primary: UrduSpeech
try:
    print("Evaluating on humairawan/UrduSpeech (streaming)...")
    ds = load_dataset("humairawan/UrduSpeech", split="test", streaming=True)
    hypotheses, references = evaluate_samples(iter(ds), n_eval, "UrduSpeech")
except Exception as e:
    print(f"UrduSpeech failed: {e}")

# Fallback: Common Voice
if not references:
    try:
        print("Trying Common Voice Urdu (streaming)...")
        ds = load_dataset(
            "mozilla-foundation/common_voice_17_0", "ur",
            split="test", streaming=True, trust_remote_code=True
        )
        def remap_cv(it):
            for s in it:
                yield {"audio": s["audio"], "text": s["sentence"]}
        hypotheses, references = evaluate_samples(
            remap_cv(iter(ds)), n_eval, "CommonVoice"
        )
    except Exception as e:
        print(f"Common Voice failed: {e}")

if references:
    w = wer(references, hypotheses)
    c = cer(references, hypotheses)
    print(f"\n{'='*50}")
    print(f"Whisper Urdu Speech ({len(references)} samples):")
    print(f"  WER: {w:.2%}")
    print(f"  CER: {c:.2%}")
    print(f"{'='*50}")
    print("\nSample predictions:")
    for ref, hyp in list(zip(references, hypotheses))[:5]:
        print(f"  REF: {ref}")
        print(f"  HYP: {hyp}\n")
else:
    print("\nNo Urdu speech dataset available (non-critical).")

del whisper_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Whisper eval done. GPU memory freed for demo.")

In [ ]:
import os, subprocess, sys

# Verify model exists
if not os.path.exists('models/risk_lora/adapter_config.json'):
    print("ERROR: No trained model found!")
    print("Complete the training cells above before running the demo.")
    sys.exit(1)

temp_path = 'models/temperature.txt'
if os.path.exists(temp_path):
    content = open(temp_path).read().strip()
    if not content:
        print("WARNING: temperature.txt is empty, writing default value 1.0")
        open(temp_path, 'w').write('1.0')
else:
    print("WARNING: temperature.txt missing, creating with default 1.0")
    os.makedirs('models', exist_ok=True)
    open(temp_path, 'w').write('1.0')

# Install gradio
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'],
                capture_output=True)

# Ensure we are in the repo root
os.chdir('/content/UrduStack')
sys.path.insert(0, '/content/UrduStack')

from playground import build_demo

print("Building unified demo...")
demo = build_demo()

print("Launching demo... (this takes 30-60 seconds)")
print("A public URL will appear below. Copy it to share.")
demo.launch(share=True)